# Análisis de gastos de diputados


### Primero abrimos los datos

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

consultorias_externas = pd.read_csv('../collected_data/ConsultoriasExternas.csv', index_col=False)
costos_operacionales = pd.read_csv('../collected_data/CostosOperacionales.csv', index_col=False)
equipo_de_apoyo = pd.read_csv('../collected_data/SupportStaff.csv', index_col=False)

costos_operacionales.head()

,Unnamed: 0,diputado,Anio,Mes,Gastos,Montos
0,0,Florcita Alarcón Rojas,2021,8,TRASPASO DESDE GASTOS OPERACIONALES A ASIGNACI...,"2,678,960"
1,1,Florcita Alarcón Rojas,2021,8,TELEFONÍA,"144,248"
2,2,Florcita Alarcón Rojas,2021,8,TRASLACIÓN,"306,147"
3,3,Florcita Alarcón Rojas,2021,8,DIFUSIÓN,0
4,4,Florcita Alarcón Rojas,2021,8,ACTIVIDADES DESTINADAS A LA INTERACCIÓN CON LA...,0


### Hay que limpiar un poco más los datos antes de poder trabajar:

In [2]:
import re

elimina_espacios_finales = lambda str: re.sub(r'\s+$', '', str)
monto_to_int = lambda x: int(re.sub(r',', '', x))

costos_operacionales['Montos'] = [monto_to_int(str) for str in costos_operacionales['Montos']]
costos_operacionales['Gastos'] = [elimina_espacios_finales(str) for str in costos_operacionales['Gastos']]
tipos_de_gasto = set(costos_operacionales['Gastos'])
tipos_de_gasto

{'ACTIVIDADES DESTINADAS A LA INTERACCIÓN CON LA COMUNIDAD',
 'ARRIENDO DE INMUEBLE',
 'ARRIENDO DE OFICINA MÓVIL',
 'ARRIENDO DE OFICINAS VIRTUALES',
 'CONSUMOS BÁSICOS',
 'CONTRATACIÓN SERVICIO DE ALMACENAMIENTO',
 'CORRESPONDENCIA',
 'COVID-19 PERSONAL DE APOYO',
 'DIFUSIÓN',
 'EQUIPAMIENTO OFICINA PARLAMENTARIA',
 'GASTOS DE MANTENCIÓN OFICINA PARLAMENTARIA (INMUEBLE)',
 'HABILITACIÓN DE SEDES PARLAMENTARIAS (CON AUTORIZACIÓN DE CRAP)',
 'MANTENCION Y REPARACIÓN DE OFICINA MÓVIL',
 'MATERIALES DE OFICINA',
 'REPARACIONES LOCATIVAS DEL INMUEBLE',
 'SEGUROS DE BIENES',
 'SERVICIOS MENORES',
 'SERVICIOS WEB',
 'TELEFONÍA',
 'TRASLACIÓN',
 'TRASPASO DESDE GASTOS OPERACIONALES A ASIGNACIÓN PERSONAL DE APOYO'}

## Veamos cuánto se gasta en cada área

In [3]:
agrupacion_por_tipo_de_gasto = costos_operacionales.groupby(by=['Gastos'])

meses = pd.PeriodIndex(year=costos_operacionales['Anio'],
                       month=costos_operacionales['Mes'], freq='M')
print(f'Período de los datos de gastos operacionales (año-mes): {meses.min()} a {meses.max()}')
print(f'La suma de gastos operacionales totales es {costos_operacionales["Montos"].sum()}')

print('El costo promedio de gasto en cada apartado es el siguiente:')
agrupacion_por_tipo_de_gasto['Montos'].mean()

Período de los datos de gastos operacionales (año-mes): 2019-01 a 2026-10
La suma de gastos operacionales totales es 27938786290
El costo promedio de gasto en cada apartado es el siguiente:


Gastos
ACTIVIDADES DESTINADAS A LA INTERACCIÓN CON LA COMUNIDAD              4.271290e+04
ARRIENDO DE INMUEBLE                                                  6.339774e+05
ARRIENDO DE OFICINA MÓVIL                                             2.559466e+04
ARRIENDO DE OFICINAS VIRTUALES                                        1.284544e+02
CONSUMOS BÁSICOS                                                      5.154519e+04
CONTRATACIÓN SERVICIO DE ALMACENAMIENTO                               2.398873e+02
CORRESPONDENCIA                                                       1.577431e+04
COVID-19 PERSONAL DE APOYO                                            4.663826e+03
DIFUSIÓN                                                              7.454687e+05
EQUIPAMIENTO OFICINA PARLAMENTARIA                                    5.147914e+04
GASTOS DE MANTENCIÓN OFICINA PARLAMENTARIA (INMUEBLE)                 2.563497e+04
HABILITACIÓN DE SEDES PARLAMENTARIAS (CON AUTORIZACIÓN DE CRAP)       4.146052e+

## Veamos quiénes gastan más en ciertos aspectos específicos:


In [4]:
import sqlite3

con = sqlite3.connect(":memory:")
try:
        costos_operacionales.to_sql('costosOperacionales', con, index=False)
        consultorias_externas.to_sql('consultoriasExternas', con, index=False)
        equipo_de_apoyo.to_sql('EquipoApoyo', con, index=False)
except ValueError:
        print('Celda ya ejecutada anteriormente')


cur = con.cursor()

query = """PRAGMA table_info(costosOperacionales);""" # muestra las columnas de la tabla

cur.execute(query)
resultado = cur.fetchall()

# # We can also close the connection if we are done with it.
# # Just be sure any changes have been committed or they will be lost.
# con.close()


In [5]:
resultado

[(0, 'Unnamed: 0', 'INTEGER', 0, None, 0),
 (1, 'diputado', 'TEXT', 0, None, 0),
 (2, 'Anio', 'INTEGER', 0, None, 0),
 (3, 'Mes', 'INTEGER', 0, None, 0),
 (4, 'Gastos', 'TEXT', 0, None, 0),
 (5, 'Montos', 'INTEGER', 0, None, 0)]

In [6]:
gasto_por_tipo_y_diputado = costos_operacionales.groupby(['Gastos', 'diputado'], as_index=False)['Montos'].sum()
top_gastos_por_diputado = gasto_por_tipo_y_diputado.groupby('Gastos')['Montos'].idxmax()
gasto_por_tipo_y_diputado.loc[top_gastos_por_diputado].set_index('Gastos')

,diputado,Montos
Gastos,,
ACTIVIDADES DESTINADAS A LA INTERACCIÓN CON LA COMUNIDAD,Catalina Del Real Mihovilovic,24635496
ARRIENDO DE INMUEBLE,Víctor Torres Jeldes,61579994
ARRIENDO DE OFICINA MÓVIL,Juan Luis Castro González,42453332
ARRIENDO DE OFICINAS VIRTUALES,Carlos Abel Jarpa Wevar,650276
CONSUMOS BÁSICOS,Giorgio Jackson Drago,7911045
CONTRATACIÓN SERVICIO DE ALMACENAMIENTO,Nicolás Noman Garrido,250000
CORRESPONDENCIA,Álvaro Carter Fernández,8352869
COVID-19 PERSONAL DE APOYO,Carlos Abel Jarpa Wevar,2931708
DIFUSIÓN,Alejandra Sepúlveda Orbenes,89176543
